# Работа с пропущенными значениями

Содержание:
1. Настройка
2. Обнаружение пропусков
3. Приведение всех типов пропусков к одному
4. Библиотека msno для визуализации пропусков
5. Обработка пропусков
6. Пример функции для общей обработки пропусков
7. Восстановление с библиотекой sklearn


## 1. Настройка

### Импорты библиотек

In [33]:
# pip install ipywidgets
import pandas as pd
import numpy as np
## Библиотека для визуализации пропущенных данных в наборах данных
import missingno as msno
## Модуль для заполнения пропусков
import sklearn 
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.impute import KNNImputer

In [23]:
# pip install missingno
# !pip install openpyxl --upgrade
# # !pip install sklearn
# !pip install --upgrade scikit-learn

### Настройки вывода

In [2]:
pd.set_option('display.float_format', '{:,.2f}'.format) 
# Уберем параметр максимального числа столбцов, что бы наглядно видеть все данные
pd.set_option('display.max_columns', None)

### Загружаем датасет

In [5]:
df = pd.read_excel('df_nan.xlsx')
df.sample(5)

,name,age,city,country,param4,param5,param6,param7,param8,param9,param10
221,Christopher Carr,26.00,Zarzecze,Poland,105.00,9.00,8.00,"109,634.00","3,040.00",10.00,107.00
11,Kimberly Miller,NaN,Kohlschwarz,Austria,118.00,12.00,12.00,"40,979.00",NaN,10.00,50.00
207,Jane Harrington,32.00,Tha Sala,Thailand,101.00,6.00,11.00,"101,852.00","3,054.00",10.00,84.00
70,Sherry Chapman,36.00,Heshi,China,113.00,8.00,8.00,"26,510.00","3,098.00",11.00,91.00
66,Stephen Howard,23.00,Tarrytown,United States,110.00,11.00,12.00,"41,408.00","3,053.00",12.00,42.00


## 2. Обнаружение пропусков

In [5]:
df.isnull().sum()

name        0
age        25
city       12
country    13
param4     16
param5     20
param6     13
param7     26
param8     21
param9     14
param10     9
dtype: int64

In [ ]:
# isna() - где ПРОПУСКИ
df.isna()                    # True где пропуски
df.isna().sum()              # количество пропусков
df.isna().mean()             # доля пропусков
df[df.isna().any(axis=1)]    # строки с пропусками

# notna() - где НЕТ пропусков
df.notna()                   # True где нет пропусков  
df.notna().sum()             # количество не-пропусков
df.notna().mean()            # доля не-пропусков
df[df.notna().all(axis=1)]   # строки без пропусков

,name,age,city,country,param4,param5,param6,param7,param8,param9,param10
0,Megan Cole,NaN,Rietheim-Weilheim,Germany,106.00,9.00,12.00,"84,889.00","3,079.00",10.00,54.00
3,David Johnson,23.00,Silleda,Spain,105.00,4.00,8.00,"30,212.00","3,078.00",NaN,53.00
4,Justin Wagner,NaN,Saint-Martin-du-Var,France,105.00,6.00,12.00,"105,592.00","3,081.00",10.00,115.00
5,Barbara Green,26.00,Saltsjobaden,Sweden,119.00,8.00,10.00,"54,484.00",NaN,11.00,31.00
8,Tom Ramirez,NaN,Cadorago,Italy,119.00,9.00,11.00,"19,873.00","3,042.00",11.00,77.00
...,...,...,...,...,...,...,...,...,...,...,...
223,Gail Williams,35.00,Stonewall,Canada,108.00,8.00,12.00,NaN,"3,090.00",11.00,45.00
224,Kathryn Sanders,NaN,Gualeguaychu,Argentina,107.00,NaN,12.00,NaN,"3,019.00",12.00,112.00
225,Alma Patton,20.00,Wanquan,China,112.00,1.00,9.00,"110,856.00",NaN,11.00,88.00
227,Derrick Pierce,53.00,Korla,China,120.00,12.00,11.00,"83,415.00","3,037.00",12.00,NaN


## 3. Приведение всех типов пропусков к одному

In [ ]:
# Частая проблема в реальных данных
mixed_data = ['apple', 'NaN', 'None', '', np.nan, None, 'orange']
s = pd.Series(mixed_data)

print(s.unique())
# ['apple' 'NaN' 'None' '' nan None 'orange']

# Нормализация пропусков
def normalize_missing(values):
    # Сначала конвертируем в строку для сравнения
    str_values = values.astype(str).str.lower()
    
    # Заменяем все варианты на np.nan
    mask = str_values.isin(['nan', 'none', 'null', ''])
    values = values.where(~mask, np.nan)
    
    # Также обрабатываем реальные NaN и None
    values = values.where(pd.notna(values), np.nan)
    
    return values

s_clean = normalize_missing(s)
print(s_clean.unique())  # ['apple' nan 'orange']

## 4. Библиотека msno для визуализации пропусков

In [ ]:
# Специальная библиотечка для пропусков
msno.bar(df)
msno.matrix(df)
msno.dendrogram(df)
# кореляии между пропусками
msno.heatmap(df)

## 5. Обработка пропусков

In [ ]:
# УДАЛЕНИЕ

# Удалить строки, где ВСЕ значения NaN
df.dropna(how='all')

# Удалить строки, где есть ХОТЯ БЫ один NaN
df.dropna() # Может сильно уменьшить выборку

# Удалить строки, где NaN в конкретной колонке
df.dropna(subset=['city'])

# Удаление столбов где превышен порог пропусков
df.dropna(axis = 1, thresh=3)
# Удалить строки, где меньше N не-NaN значений
df.dropna(thresh=len(df.columns)*0.9) # Оставить строки с >=90% данных

#УДАЛЕНИЕ СТОЛБЦОВ
threshold = 0.7 # Удалить колонки, где более 70% пропусков
df.loc[:, df.isnull().mean() < threshold]

,name,age,city,country,param4,param5,param6,param7,param8,param9,param10
0,Megan Cole,NaN,Rietheim-Weilheim,Germany,106.00,9.00,12.00,"84,889.00","3,079.00",10.00,54.00
1,Lena Holmes,56.00,Lasbek,Germany,106.00,11.00,11.00,"76,197.00","3,049.00",12.00,74.00
2,Cheryl West,23.00,Arsaki,Russian Federation,113.00,6.00,9.00,"72,897.00","3,024.00",12.00,7.00
3,David Johnson,23.00,Silleda,Spain,105.00,4.00,8.00,"30,212.00","3,078.00",NaN,53.00
4,Justin Wagner,NaN,Saint-Martin-du-Var,France,105.00,6.00,12.00,"105,592.00","3,081.00",10.00,115.00
...,...,...,...,...,...,...,...,...,...,...,...
225,Alma Patton,20.00,Wanquan,China,112.00,1.00,9.00,"110,856.00",NaN,11.00,88.00
226,Donna Chavez,47.00,Baqat al Hatab,Occupied Palestinian Territory,106.00,10.00,8.00,"100,150.00","3,027.00",10.00,62.00
227,Derrick Pierce,53.00,Korla,China,120.00,12.00,11.00,"83,415.00","3,037.00",12.00,NaN
228,Juanita Barnes,24.00,Hukeng,China,119.00,11.00,9.00,"57,129.00","3,003.00",10.00,18.00


In [ ]:
# ЗАПОЛНЕНИЕ
df.fillna(0)
# Замена пропущенных значений средними значениями
df['age'] = df['age'].fillna(df['age'].mean(), inplace=True)
df['param4'] = df['param4'].fillna(df['param4'].median(), inplace=True)
df['city'] = df['city'].fillna(df['city'].mode()[0]) # Мода (для категориальных)

#Заполнение пропущенных значений на основе дополнительной информации
df.loc[df['city'] == 'Lasbek', 'param8'] = 17625

df['param9'].fillna(method='ffill') # Заполнить предыдущим значением 
df['param10'].fillna(method='bfill') # Заполнить следующим значением

# Категориальные данные
df['country'] = df['country'].fillna('Unknown') # Создать новую категорию "Неизвестно"

# Присвоение пропускам специальной категории
df.fillna(999999)

In [7]:
## Проставление метки
df['param9_IS_MISSING'] = df['param9'].isnull().astype(int)

## 6. Пример функции для общей обработки пропусков


In [ ]:
def handle_missing_data(df):
    df_clean = df.copy()

    # Удаляем столбец, если пропусков >80%
    df_clean = df_clean.loc[:, df_clean.isnull().mean() < 0.8]

    # Для категориальных: создаем индикатор и заполняем "Unknown"
    cat_cols = df_clean.select_dtypes(include='object').columns
    for col in cat_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col+'_MISSING'] = df_clean[col].isnull().astype(int)
            df_clean[col] = df_clean[col].fillna('Unknown')

    # Для числовых: создаем индикатор и заполняем медианой
    num_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns
    for col in num_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col+'_MISSING'] = df_clean[col].isnull().astype(int)
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)

    return df_clean

## 7. Восстановление с библиотекой sklearn

In [ ]:
# Восстановление по одному столбцу
imp_mean =SimpleImputer(missing_values=np.nan, strategy='mean') ## заполнение 
imp_mean
# imp_mean.fit(df[['param8']])
# imp_mean.transform(df[['param8']])

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [ ]:
## Восстановление по нескольким столбцах
imp = IterativeImputer(max_iter=10, random_state=0)
imp.fit([[1, 2], [3, 6], [4, 8], [np.nan, 3], [7, np.nan]])
IterativeImputer(random_state=0)
X_test = [[np.nan, 2], [6, np.nan], [np.nan, 6]]
# the model learns that the second feature is double the first
print(np.round(imp.transform(X_test)))

[[ 1.  2.]
 [ 6. 12.]
 [ 3.  6.]]


In [34]:
## По блтжайшим соседям
nan = np.nan
X = [[1, 2, nan], [3, 4, 3], [nan, 6, 5], [8, 8, 7]]
imputer = KNNImputer(n_neighbors=2, weights="uniform")
imputer.fit_transform(X)

array([[1. , 2. , 4. ],
       [3. , 4. , 3. ],
       [5.5, 6. , 5. ],
       [8. , 8. , 7. ]])